# Discovery - Limpieza de Datos

## Inspeción preliminar

In [60]:
import pandas as pd
import pathlib
import re
import unicodedata
import ast
import numpy as np

In [41]:
# Ruta recomendada para compartir el proyecto sin versionar datos sensibles.
path_todos_registros = pathlib.Path("../data_raw/todos_registros.xlsx")

# Fallback para el entorno local original del proyecto.
if not path_todos_registros.exists():
    path_todos_registros = pathlib.Path(r"D:\Open_Vzla_SOS_Calidad_de_Datos\aevscraping\datos_consolidados\todos_registros.xlsx")

df_raw = pd.read_excel(path_todos_registros)

# Conservamos el scraping original intacto y trabajamos siempre sobre una copia.
df_clean = df_raw.copy()
df_clean.head()


,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente
0,7410076847799689,José Gregorio Rojas Bello,No registrado,NaN,La Guaira,+584142504728,No sabemos d el y su familia padre esposa hijos,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/7ec6...,2026-06-28T17:50:42.114Z,2026-06-28T17:50:42.114Z,No,venezuelatebusca
1,2590484473230077,Hector Reyes,No registrado,NaN,San Agustin,+584124989844,Acabo de ver un vídeo de WhatsApp donde tienen...,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/cc73...,2026-06-28T17:50:34.923Z,2026-06-28T17:50:34.923Z,No,venezuelatebusca
2,152612323455194,Neiderlyn Alfonso,No registrado,6.0,Tanaguarenas la Guaira los cocos,+584241592570,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/ba9b...,2026-06-28T17:49:37.868Z,2026-06-28T17:49:37.868Z,Sí,venezuelatebusca
3,2394053969385436,Samuel Vicente Cortez Olivier,37099131,11.0,"Conjunto Residencial Caribe, la Guaira",+584127387227,Pantalon azul franela,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/53e6...,2026-06-28T17:48:07.144Z,2026-06-28T17:48:07.144Z,Sí,venezuelatebusca
4,6937486286932653,Amir Infante Galván,No registrado,16.0,La guaira,+584162443424,NaN,Desaparecido,NaN,NaN,NaN,NaN,2026-06-28T17:48:04.530Z,2026-06-28T17:48:04.530Z,Sí,venezuelatebusca


In [80]:
# Filas y columnas del DataFrame
df_clean.shape

(241463, 16)

In [81]:
# Información general del DataFrame
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    241463 non-null  int64              
 1   Nombre                241463 non-null  str                
 2   Cédula                241463 non-null  str                
 3   Edad                  140396 non-null  float64            
 4   Última Ubicación      228992 non-null  str                
 5   Teléfono Contacto     101095 non-null  str                
 6   Observaciones         162535 non-null  str                
 7   Estado                241463 non-null  str                
 8   Ubicación Encontrado  14678 non-null   str                
 9   Encontrado Por        16091 non-null   str                
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              129962 non-null  str                
 12 

## Normalización

In [84]:
# Función para quitar acentos
def quitar_acentos(texto):
    if pd.isna(texto):
        return pd.NA

    return ''.join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

In [85]:
# Función para normalizar texto
# Centralizamos la limpieza básica para aplicar el mismo criterio a todas las columnas textuales.
def normalizar_texto(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto)
    texto = quitar_acentos(texto)
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)

    return texto

In [86]:
# Normalizar columnas de texto
columnas_texto = [
    "Nombre",
    "Última Ubicación",
    "Teléfono Contacto",
    "Observaciones",
    "Estado",
    "Ubicación Encontrado",
    "Encontrado Por",
    "URL Foto",
    "Fuente"
]

for col in columnas_texto:
    df_clean[col] = df_clean[col].apply(normalizar_texto)

In [87]:
# Normalizar la cédula
df_clean["Cédula"] = (
    df_clean["Cédula"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

In [88]:
df_clean["Teléfono Contacto"] = (
    df_clean["Teléfono Contacto"]
    .str.replace(r"\D", "", regex=True)
)

In [89]:
# Convertir fechas
df_clean["Fecha Registro"] = pd.to_datetime(
    df_clean["Fecha Registro"],
    errors="coerce"
)

df_clean["Fecha Actualización"] = pd.to_datetime(
    df_clean["Fecha Actualización"],
    errors="coerce"
)

In [90]:
# Edad
df_clean["Edad"] = pd.to_numeric(
    df_clean["Edad"],
    errors="coerce"
)

In [92]:
# Es Menor
df_clean["Es Menor"] = (
    df_clean["Es Menor"]
    .str.strip()
    .str.title()
)


In [93]:
# Estado
df_clean["Estado"] = (
    df_clean["Estado"]
    .str.strip()
    .str.title()
)

In [94]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    241463 non-null  int64              
 1   Nombre                241463 non-null  str                
 2   Cédula                241463 non-null  str                
 3   Edad                  140396 non-null  float64            
 4   Última Ubicación      228992 non-null  str                
 5   Teléfono Contacto     101095 non-null  str                
 6   Observaciones         162535 non-null  str                
 7   Estado                241463 non-null  str                
 8   Ubicación Encontrado  14678 non-null   str                
 9   Encontrado Por        16091 non-null   str                
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              129962 non-null  str                
 12 

## Validación de los datos

In [95]:
df_validated = df_clean.copy()

In [96]:
# Validar cédulas
mask = (
    df_validated["Cédula"]
    .str.fullmatch(r"\d{7,8}", na=False)
)

df_validated.loc[~mask, "Cédula"] = pd.NA

In [97]:
df_validated["Cédula"].str.len().value_counts(dropna=False)

Cédula
NaN    230767
8.0      8065
7.0      2631
Name: count, dtype: int64

In [98]:
# Validar teléfonos
mask = (
    df_validated["Teléfono Contacto"]
    .str.fullmatch(r"\d{10,15}", na=False)
)

df_validated.loc[~mask, "Teléfono Contacto"] = pd.NA

In [99]:
# Validar edad
mask = df_validated["Edad"].between(0, 110)

df_validated.loc[~mask, "Edad"] = pd.NA

In [100]:
# Validar URLs
mask = (
    df_validated["URL Foto"]
    .str.match(r"^https?://", na=False)
)

df_validated.loc[~mask, "URL Foto"] = pd.NA

In [101]:
# Validar Estado
df_validated["Estado"].value_counts(dropna=False)

Estado
Desaparecido    203737
Localizado       37726
Name: count, dtype: int64

In [102]:
estados_validos = {
    "Localizado",
    "Desaparecido"
}

mask = df_validated["Estado"].isin(estados_validos)

df_validated.loc[~mask, "Estado"] = pd.NA

In [103]:
# Cambiar el tipo str a string para manejo de los valores faltantes para evitar inconsistencias con NaN, None y cadenas vacías.
columnas_texto = [
    "Nombre",
    "Cédula",
    "Última Ubicación",
    "Teléfono Contacto",
    "Observaciones",
    "Estado",
    "Ubicación Encontrado",
    "Encontrado Por",
    "URL Foto",
    "Es Menor",
    "Fuente"
]

df_validated[columnas_texto] = (
    df_validated[columnas_texto]
    .astype("string")
)

In [104]:
df_validated.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    241463 non-null  int64              
 1   Nombre                241463 non-null  string             
 2   Cédula                10696 non-null   string             
 3   Edad                  140391 non-null  float64            
 4   Última Ubicación      228992 non-null  string             
 5   Teléfono Contacto     86550 non-null   string             
 6   Observaciones         162535 non-null  string             
 7   Estado                241463 non-null  string             
 8   Ubicación Encontrado  14678 non-null   string             
 9   Encontrado Por        16091 non-null   string             
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              129962 non-null  string             
 12 

## Entities

In [105]:
df_entities = df_validated.copy()

In [ ]:
# Creamos un identificador técnico por fila para poder unir registros que comparten datos de identidad.
df_entities["fila_id"] = np.arange(len(df_entities))

In [ ]:
# Estructura simple de conjuntos disjuntos para fusionar filas que representan a la misma persona.
padre = np.arange(len(df_entities))


def buscar_raiz(fila_id):
    while padre[fila_id] != fila_id:
        padre[fila_id] = padre[padre[fila_id]]
        fila_id = padre[fila_id]

    return fila_id


def unir_filas(fila_ids):
    fila_ids = list(fila_ids)
    fila_base = buscar_raiz(fila_ids[0])

    for fila_id in fila_ids[1:]:
        raiz = buscar_raiz(fila_id)

        if raiz != fila_base:
            padre[raiz] = fila_base

In [ ]:
# Cédula y teléfono son claves fuertes: si dos filas comparten alguna de estas claves,
# probablemente hablan de la misma persona y deben quedar en la misma entidad.
claves_fuertes = ["Cédula", "Teléfono Contacto"]

for columna in claves_fuertes:
    grupos = df_entities.dropna(subset=[columna]).groupby(columna)["fila_id"]

    for _, fila_ids in grupos:
        unir_filas(fila_ids)

In [ ]:
# El nombre solo es una clave débil: no alcanza por sí solo para identificar una persona.
# Lo usamos junto con edad y última ubicación para reducir el riesgo de fusionar homónimos.
mask = (
    df_entities["Nombre"].notna() &
    df_entities["Edad"].notna() &
    df_entities["Última Ubicación"].notna()
)

grupos = df_entities.loc[mask].groupby(["Nombre", "Edad", "Última Ubicación"])["fila_id"]

for _, fila_ids in grupos:
    unir_filas(fila_ids)

In [ ]:
# Convertimos cada grupo conectado en un entity_id compacto y estable dentro del notebook.
raices = pd.Series(df_entities["fila_id"]).apply(buscar_raiz)

df_entities["entity_id"] = pd.factorize(raices)[0]
df_entities = df_entities.drop(columns="fila_id")

df_entities["entity_id"].isna().sum()

In [ ]:
# Resumen rápido de entidades construidas
resumen_entities = pd.Series({
    "registros": len(df_entities),
    "entidades": df_entities["entity_id"].nunique(),
    "entidades_con_multiples_registros": (df_entities.groupby("entity_id").size() > 1).sum(),
})

resumen_entities